### Note
A Regime Filter is a trading technical analysis tool to identify the current market(eg bullish, bearish) to adjust strategies accordingly to prevent a strategy in an environment that it wasn't made for.
In this case, we want to avoid trading in high volatility regimes where mean reversion breaks down. (To do this we use an indicator function)

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

price_data = pd.read_csv("../data/prices.csv", index_col = 0, parse_dates = True).dropna()
price_data.head()

,AAPL,AMD,BAC,CVX,JPM,KO,MSFT,NVDA,PEP,XOM
Date,,,,,,,,,,
2023-01-03,123.096024,64.019997,30.974270,151.901291,124.928711,57.519154,233.452820,14.300685,162.498108,95.434189
2023-01-04,124.365662,64.660004,31.556595,150.286133,126.093643,57.491737,223.240829,14.734250,162.099548,95.711967
2023-01-05,123.046814,62.330002,31.491901,152.992584,126.065750,56.833855,216.624466,14.250736,160.405838,97.853439
2023-01-06,127.574188,63.959999,31.806168,154.145020,128.478058,57.930328,219.177444,14.844140,164.028809,99.036171
2023-01-09,128.095840,67.239998,31.325510,152.940186,127.947151,57.208481,221.311462,15.612370,162.425613,97.190392


In [29]:
# Adding SPY to price_data
import yfinance as yf

spy = yf.download("SPY", start=price_data.index.min(), end=price_data.index.max())["Close"]

price_data["SPY"] = spy

[*********************100%***********************]  1 of 1 completed


In [30]:
# Hedge Ratio
t1,t2 = "XOM", "CVX"

entry_z = 1.5
z_window = 20
cost = 0.0005

x = np.log(price_data[t1])    
y = np.log(price_data[t2])

X = add_constant(x)
model = OLS(y,X).fit()
beta = model.params[1]

print("Hedge ratio beta:", beta)

Hedge ratio beta: 0.5326573018875291


C:\Users\yogst\AppData\Local\Temp\ipykernel_6172\2274502009.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]


In [31]:
# Spread and Z-score
spread = y - beta*x

mean = spread.rolling(z_window).mean()
std = spread.rolling(z_window).std()
zscore = (spread-mean)/std

In [32]:
# Trading Signals
position = pd.Series(0, index = spread.index)

position[zscore > entry_z] = -1
position[zscore < -entry_z] = 1

position = position.ffill().fillna(0)

In [33]:
# Compute Strategy Returns
spread_ret = spread.diff()
strategy_ret = position.shift(1) * spread_ret

trades = position.diff().abs()
strategy_ret = strategy_ret - trades * cost

strategy_ret = strategy_ret.fillna(0)

In [37]:
# Computer Market volatility
market_returns = np.log(price_data["SPY"]).diff()
market_vol = market_returns.rolling(20).std() 

# market volatility = rolling std of log returns

In [45]:
# Define a Regime Threshold
vol_threshold = market_vol.quantile(0.75)
low_vol_regime = market_vol < vol_threshold # creates a boolean series, true if low volatility, vice versa
# conditioning our filter on volatility

In [40]:
# Apply Regime Filter
filtered_ret = strategy_ret * low_vol_regime # boolean acts as a indicator variable
filtered_ret = filtered_ret.fillna(0)

In [43]:
# Performance function
def performance_stats(returns):
    sharpe = returns.mean() / returns.std() * np.sqrt(252)

    equity = (1+ returns).cumprod()
    peak = equity.cummax()
    drawdown = (equity - peak) / peak
    max_dd = drawdown.min()

    cagr = equity.iloc[-1]**(252/len(returns))-1

    return sharpe, cagr, max_dd

In [44]:
# Results
sharpe_raw, cagr_raw, dd_raw = performance_stats(strategy_ret)
sharpe_filtered, cagr_filtered, dd_filtered = performance_stats(filtered_ret)

print("=== Without Regime Filter ===")
print("Sharpe:", sharpe_raw)
print("CAGR:", cagr_raw)
print("Max DD:", dd_raw)

print("\n=== With Regime Filter ===")
print("Sharpe:", sharpe_filtered)
print("CAGR:", cagr_filtered)
print("Max DD:", dd_filtered)

=== Without Regime Filter ===
Sharpe: -0.6355689958026408
CAGR: -0.05214803218898123
Max DD: -0.20984923995851454

=== With Regime Filter ===
Sharpe: -0.410216035377905
CAGR: -0.030062954240929307
Max DD: -0.15516630492247993


### Interpretation
Note that with the Regime Filter, sharpe, CAGR and max drawdown has improved, depsite the fact that all statistics are negative and may not be profitable to trade ie have not found alpha.